In [ ]:
print("Activated")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder

### Transform

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

### Load the data

In [ ]:
train_dataset_path = r"C:\Users\TAQICOMPUTERS\Desktop\Face-Mask-Detector\Train"
test_dataset_path = r"C:\Users\TAQICOMPUTERS\Desktop\Face-Mask-Detector\Test"
val_dataset_path = r"C:\Users\TAQICOMPUTERS\Desktop\Face-Mask-Detector\Validation"

train_data = ImageFolder(train_dataset_path, transform=transform)
test_data = ImageFolder(test_dataset_path, transform=transform)
val_data = ImageFolder(val_dataset_path, transform=transform)

print(train_data.classes)

### Data Loaders

In [ ]:
train_loader = DataLoader(train_data, batch_size=85, shuffle=True)
test_loader = DataLoader(test_data, batch_size=85, shuffle=False)
val_loader = DataLoader(val_data, batch_size=85, shuffle=False)

### Model Architecture

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size= 3, padding= 0),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size= 3, padding= 0),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size= 3, padding= 0),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(14 * 14 * 128, 2256),
            nn.ReLU(),
            nn.Linear(2256, 2)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x

### Build Model

In [ ]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Training loop

In [ ]:
epochs = 10
model.train()

print("------Training Started------")
for epoch in range(epochs):
    epoch_train_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item()
    
    avg_loss = epoch_train_loss / len(train_loader)
    print(f"Epoch {epoch + 1} / {epochs} - Loss: {avg_loss:.4f}")

In [ ]:
model.eval() 
correct_labels = 0
total_labels = 0

with torch.no_grad(): 
    for images, labels in val_loader:
        outputs = model(images)                
        _, predicted = torch.max(outputs, 1)   
        correct_labels += (predicted == labels).sum().item()  
        total_labels += labels.size(0)                        

accuracy = (correct_labels / total_labels) * 100
print(f"\nValidation Accuracy: {accuracy:.2f}%")

torch.save(model.state_dict(), 'face_mask_detector.pth')
print("Model saved successfully as 'face_mask_detector.pth'!")